# Creating an ndtbl file with Python

This notebook creates a small `.ndtbl` file with one uniformly spaced axis and one explicit non-uniform axis. The same pattern works for larger tables: define the axes, build a NumPy array with shape `axis_0 x axis_1 x ... x field`, wrap it in a `FieldGroup`, and write it with `write_group`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import ndtbl

## Define the axes

`UniformAxis(min, max, size)` stores a compact, evenly spaced coordinate axis. `ExplicitAxis([...])` stores all coordinates and is useful when the spacing is not uniform. Explicit-axis coordinates must be strictly increasing.

In [ ]:
temperature_axis = ndtbl.UniformAxis(min=300.0, max=1200.0, size=4)
mixture_fraction_axis = ndtbl.ExplicitAxis([0.0, 0.02, 0.10, 0.35, 1.0])

temperature = temperature_axis.coordinates()
mixture_fraction = mixture_fraction_axis.coordinates()

temperature_grid, mixture_fraction_grid = np.meshgrid(
    temperature,
    mixture_fraction,
    indexing="ij",
)

temperature, mixture_fraction

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.scatter(mixture_fraction_grid, temperature_grid, color="black", s=10)
ax.scatter(
    mixture_fraction[2],
    temperature[1],
    color="tab:red",
    s=50,
    label="values[1, 2, :]",
)

for value in mixture_fraction:
    ax.axvline(value, color="0.85", linewidth=1, zorder=0)
for value in temperature:
    ax.axhline(value, color="0.85", linewidth=1, zorder=0)

ax.set_xlabel("mixture_fraction, explicit non-uniform axis")
ax.set_ylabel("temperature, uniform axis")
ax.set_title("Grid points stored in the ndtbl payload")
ax.legend()
plt.show()

## Build field values

Here we create two fields, `enthalpy` and `source_term`, over the 2D grid. The final array shape is `(temperature_axis.size, mixture_fraction_axis.size, 2)`, where the last dimension stores the fields in `field_names` order.

In [ ]:
enthalpy = 1.0e3 + 0.25 * temperature_grid + 40.0 * mixture_fraction_grid
source_term = (
    2.0 * mixture_fraction_grid * (1.0 - mixture_fraction_grid)
    + 1.0e-4 * temperature_grid
)

values = np.stack((enthalpy, source_term), axis=-1).astype(np.float64)

group = ndtbl.FieldGroup(
    axes=(temperature_axis, mixture_fraction_axis),
    field_names=("enthalpy", "source_term"),
    values=values,
)

group.axis_sizes, group.field_names, group.values.shape

## Write and read the file

`write_group` serializes the complete table. `read_metadata` reads only the lightweight description, while `read_group` loads metadata and payload values.

In [ ]:
output_path = Path("mixed_axes_example.ndtbl")

ndtbl.write_group(output_path, group)

metadata = ndtbl.read_metadata(output_path)
loaded = ndtbl.read_group(output_path)

(
    metadata.dimension,
    metadata.axis_sizes,
    metadata.field_names,
    metadata.dtype_name,
)

## Use the loaded values

Values are indexed by zero-based axis indices followed by the field index. This reads the two field values at the grid point `temperature=600.0` and `mixture_fraction=0.10`.

In [ ]:
temperature_index = 1
mixture_fraction_index = 2

point_values = dict(
    zip(
        loaded.field_names,
        loaded.values[temperature_index, mixture_fraction_index, :],
    )
)

point_coordinates = {
    "temperature": loaded.axes[0].coordinates()[temperature_index],
    "mixture_fraction": loaded.axes[1].coordinates()[mixture_fraction_index],
}

point_coordinates, point_values